# 08 – Ingeniería de Variables

En este notebook se construyen variables derivadas a partir del dataset procesado:

`../data/processed/training_dataset.parquet`

## Objetivos

- Limpiar columnas poco útiles para el modelo base
- Ordenar correctamente el dataset por vaca y fecha
- Crear variables temporales por vaca
- Generar rezagos (`lags`) de producción
- Generar promedios móviles (`rolling features`)
- Construir variables temporales del calendario
- Crear algunas variables derivadas de clima
- Preparar un dataset enriquecido para modelado

> Nota: este notebook está pensado como una primera versión del pipeline de ingeniería de variables.  
> Se priorizan variables robustas y con buena cobertura.


## 1. Importaciones y configuración

Se cargan las librerías principales para manipulación tabular y análisis numérico.


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)


## 2. Carga del dataset base

Aquí se carga el dataset producido en la etapa anterior.  
Este será el punto de partida para construir nuevas variables.


In [2]:
INPUT_PATH = "../data/processed/training_dataset.parquet"

df = pd.read_parquet(INPUT_PATH)

print("Shape original:", df.shape)
display(df.head())


Shape original: (9814, 31)


,cow_id,date,produccion_kg,di,dd,ti,td,ordenos_dia,duracion_total_min,intervalo_ordeno_prom_min,ubre,destino_leche,ms,source_file,rumia_min,days_in_milk,lactation_age,weekday,month,group_id,source_file_rum,temperature_c,humidity_pct,pressure_msl,rain_mm,wind_speed,eventos_pdf_count,eventos_pdf_text,year,day,weekday_date
0,1204,2025-01-01,16.68,5.16,4.64,0.00,6.88,1,0.0,NaN,0,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,1,NaN,NaN,14.266667,43.291667,1019.283333,0.000000,11.150000,NaN,NaN,2025,1,2
1,1204,2025-01-02,17.91,2.65,4.44,5.75,5.07,2,0.0,NaN,1,Divert 3,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,1,NaN,NaN,13.404167,64.916667,1022.504167,0.000000,13.187500,NaN,NaN,2025,2,3
2,1204,2025-01-03,25.36,6.26,6.52,2.77,9.81,2,0.0,NaN,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,1,NaN,NaN,12.791667,73.000000,1023.987500,0.000000,16.045833,NaN,NaN,2025,3,4
3,1204,2025-01-04,16.71,4.13,3.89,2.96,5.73,1,0.0,NaN,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,1,NaN,NaN,14.333333,70.750000,1020.816667,0.029167,7.800000,NaN,NaN,2025,4,5
4,1204,2025-01-05,25.04,5.85,5.94,4.85,8.40,2,0.0,NaN,1,Tanque,VMS 1,Producciones de leche1204.xls,NaN,NaN,NaN,NaN,1,NaN,NaN,14.333333,58.458333,1018.504167,0.000000,8.445833,NaN,NaN,2025,5,6


### Comentarios y observaciones

Antes de crear variables nuevas conviene confirmar:

- que `cow_id` y `date` existan
- que `date` sea interpretable como fecha
- que `produccion_kg` esté disponible
- que no existan duplicados inesperados en `cow_id` + `date`


## 3. Normalización básica de tipos y orden

Para construir variables temporales correctamente, es indispensable que:

- `date` esté en formato de fecha
- el dataframe esté ordenado por `cow_id` y `date`


In [3]:
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(["cow_id", "date"]).reset_index(drop=True)

print("Shape después de ordenar:", df.shape)
display(df[["cow_id", "date"]].head(10))


Shape después de ordenar: (9814, 31)


,cow_id,date
0,1204,2025-01-01
1,1204,2025-01-02
2,1204,2025-01-03
3,1204,2025-01-04
4,1204,2025-01-05
5,1204,2025-01-06
6,1204,2025-01-07
7,1204,2025-01-08
8,1204,2025-01-09
9,1204,2025-01-10


### Comentarios y observaciones

El orden por vaca y fecha es esencial para que los `shift()` y los `rolling()` generen variables correctas.


## 4. Revisión de duplicados por vaca y fecha

Antes de construir rezagos, se revisa si existen múltiples filas para la misma vaca en la misma fecha.


In [4]:
dups = df.duplicated(subset=["cow_id", "date"]).sum()

print("Duplicados por cow_id + date:", dups)

if dups > 0:
    display(df[df.duplicated(subset=["cow_id", "date"], keep=False)].sort_values(["cow_id", "date"]).head(20))


Duplicados por cow_id + date: 0


### Comentarios y observaciones

Si aparecen duplicados, hay que decidir si:

- se agregan
- se eliminan
- o se conserva una sola observación por vaca y fecha

Para este notebook se asume que el dataset ya viene a nivel diario por vaca.


## 5. Revisión de valores faltantes

Se vuelve a revisar la disponibilidad de las columnas para decidir qué variables sí conviene usar en la versión base del modelo.


In [5]:
missing = df.isna().mean().sort_values(ascending=False)

display(missing)


intervalo_ordeno_prom_min    1.000000
rumia_min                    0.997249
source_file_rum              0.997147
group_id                     0.997147
weekday                      0.997147
lactation_age                0.997147
days_in_milk                 0.997147
eventos_pdf_count            0.984920
eventos_pdf_text             0.984920
td                           0.000000
cow_id                       0.000000
di                           0.000000
dd                           0.000000
date                         0.000000
produccion_kg                0.000000
source_file                  0.000000
ms                           0.000000
destino_leche                0.000000
ubre                         0.000000
ordenos_dia                  0.000000
duracion_total_min           0.000000
ti                           0.000000
month                        0.000000
humidity_pct                 0.000000
temperature_c                0.000000
rain_mm                      0.000000
pressure_msl

### Comentarios y observaciones

En esta etapa normalmente se detecta que algunas columnas tienen cobertura casi nula.  
Esas columnas pueden excluirse del modelo base para evitar ruido innecesario.


## 6. Eliminación de columnas con demasiados faltantes

En esta primera versión se eliminan columnas con cobertura muy baja o con utilidad marginal para el modelo base.

La lógica aquí es pragmática:

- conservar señales robustas
- evitar columnas casi vacías
- simplificar el dataset de entrada al modelo


In [6]:
drop_cols = [
    "intervalo_ordeno_prom_min",
    "source_file_rum",
]

drop_cols = [col for col in drop_cols if col in df.columns]

df = df.drop(columns=drop_cols)

print("Columnas eliminadas:", drop_cols)
print("Shape actual:", df.shape)


Columnas eliminadas: ['intervalo_ordeno_prom_min', 'source_file_rum']
Shape actual: (9814, 29)


### Comentarios y observaciones

Variables como `rumia_min`, `days_in_milk` o `lactation_age` pueden conservarse por ahora si existen, pero su uso real dependerá de su cobertura.

Más adelante podrían:

- imputarse
- reconstruirse desde otra fuente
- o excluirse en la etapa final de modelado


## 7. Variables básicas del calendario

Se crean variables temporales derivadas de la fecha.

Estas variables permiten capturar patrones como:

- estacionalidad
- comportamiento semanal
- cambios mensuales


In [7]:
df["anio"] = df["date"].dt.year
df["mes"] = df["date"].dt.month
df["dia_mes"] = df["date"].dt.day
df["dia_semana"] = df["date"].dt.weekday
df["es_fin_de_semana"] = df["dia_semana"].isin([5, 6]).astype(int)
df["dia_del_anio"] = df["date"].dt.dayofyear

display(df[["date", "anio", "mes", "dia_mes", "dia_semana", "es_fin_de_semana", "dia_del_anio"]].head())


,date,anio,mes,dia_mes,dia_semana,es_fin_de_semana,dia_del_anio
0,2025-01-01,2025,1,1,2,0,1
1,2025-01-02,2025,1,2,3,0,2
2,2025-01-03,2025,1,3,4,0,3
3,2025-01-04,2025,1,4,5,1,4
4,2025-01-05,2025,1,5,6,1,5


### Comentarios y observaciones

Estas variables son simples pero útiles.  
En muchos problemas temporales capturan parte de la variación sistemática del proceso.


## 8. Variables cíclicas del calendario

Las variables como mes o día de la semana tienen naturaleza cíclica.  
Por eso se transforman a seno y coseno.

Esto evita tratar diciembre y enero como si estuvieran "lejos" entre sí.


In [8]:
df["mes_sin"] = np.sin(2 * np.pi * df["mes"] / 12)
df["mes_cos"] = np.cos(2 * np.pi * df["mes"] / 12)

df["dia_semana_sin"] = np.sin(2 * np.pi * df["dia_semana"] / 7)
df["dia_semana_cos"] = np.cos(2 * np.pi * df["dia_semana"] / 7)

display(df[["mes", "mes_sin", "mes_cos", "dia_semana", "dia_semana_sin", "dia_semana_cos"]].head())


,mes,mes_sin,mes_cos,dia_semana,dia_semana_sin,dia_semana_cos
0,1,0.5,0.866025,2,0.974928,-0.222521
1,1,0.5,0.866025,3,0.433884,-0.900969
2,1,0.5,0.866025,4,-0.433884,-0.900969
3,1,0.5,0.866025,5,-0.974928,-0.222521
4,1,0.5,0.866025,6,-0.781831,0.623490


### Comentarios y observaciones

Estas transformaciones suelen ayudar bastante en modelos que no capturan periodicidad por sí solos.


## 9. Rezagos de producción por vaca

Ahora se crean variables históricas de `produccion_kg` usando `shift()` agrupado por vaca.

Estas variables responden a una idea central:

> la producción de hoy suele depender en parte de la producción reciente de esa misma vaca


In [9]:
group = df.groupby("cow_id", group_keys=False)

df["produccion_lag_1"] = group["produccion_kg"].shift(1)
df["produccion_lag_2"] = group["produccion_kg"].shift(2)
df["produccion_lag_3"] = group["produccion_kg"].shift(3)
df["produccion_lag_7"] = group["produccion_kg"].shift(7)

display(
    df[
        ["cow_id", "date", "produccion_kg", "produccion_lag_1", "produccion_lag_2", "produccion_lag_3", "produccion_lag_7"]
    ].head(15)
)


,cow_id,date,produccion_kg,produccion_lag_1,produccion_lag_2,produccion_lag_3,produccion_lag_7
0,1204,2025-01-01,16.68,NaN,NaN,NaN,NaN
1,1204,2025-01-02,17.91,16.68,NaN,NaN,NaN
2,1204,2025-01-03,25.36,17.91,16.68,NaN,NaN
3,1204,2025-01-04,16.71,25.36,17.91,16.68,NaN
4,1204,2025-01-05,25.04,16.71,25.36,17.91,NaN
5,1204,2025-01-06,10.39,25.04,16.71,25.36,NaN
6,1204,2025-01-07,23.96,10.39,25.04,16.71,NaN
7,1204,2025-01-08,24.79,23.96,10.39,25.04,16.68
8,1204,2025-01-09,32.44,24.79,23.96,10.39,17.91
9,1204,2025-01-10,28.80,32.44,24.79,23.96,25.36


### Comentarios y observaciones

Estas variables son de las más importantes en problemas de predicción temporal.

Hay que recordar que los primeros días de cada vaca tendrán valores faltantes en estos rezagos, lo cual es completamente normal.


## 10. Cambios diarios de producción

Además de los rezagos, también es útil medir cuánto cambia la producción respecto a días anteriores.


In [10]:
df["delta_produccion_1d"] = df["produccion_kg"] - df["produccion_lag_1"]
df["delta_produccion_3d"] = df["produccion_kg"] - df["produccion_lag_3"]

display(
    df[
        ["cow_id", "date", "produccion_kg", "produccion_lag_1", "delta_produccion_1d", "produccion_lag_3", "delta_produccion_3d"]
    ].head(15)
)


,cow_id,date,produccion_kg,produccion_lag_1,delta_produccion_1d,produccion_lag_3,delta_produccion_3d
0,1204,2025-01-01,16.68,NaN,NaN,NaN,NaN
1,1204,2025-01-02,17.91,16.68,1.23,NaN,NaN
2,1204,2025-01-03,25.36,17.91,7.45,NaN,NaN
3,1204,2025-01-04,16.71,25.36,-8.65,16.68,0.03
4,1204,2025-01-05,25.04,16.71,8.33,17.91,7.13
5,1204,2025-01-06,10.39,25.04,-14.65,25.36,-14.97
6,1204,2025-01-07,23.96,10.39,13.57,16.71,7.25
7,1204,2025-01-08,24.79,23.96,0.83,25.04,-0.25
8,1204,2025-01-09,32.44,24.79,7.65,10.39,22.05
9,1204,2025-01-10,28.80,32.44,-3.64,23.96,4.84


### Comentarios y observaciones

Estas diferencias pueden capturar:

- caídas abruptas
- recuperaciones rápidas
- cambios anómalos en la dinámica de una vaca


## 11. Promedios móviles de producción por vaca

Se crean promedios móviles usando solo información del pasado.

> Importante: se usa `shift(1)` antes del `rolling()` para evitar fuga de información.


In [11]:
df["produccion_roll_mean_3"] = group["produccion_kg"].shift(1).rolling(window=3).mean()
df["produccion_roll_mean_7"] = group["produccion_kg"].shift(1).rolling(window=7).mean()

df["produccion_roll_std_3"] = group["produccion_kg"].shift(1).rolling(window=3).std()
df["produccion_roll_std_7"] = group["produccion_kg"].shift(1).rolling(window=7).std()

display(
    df[
        [
            "cow_id", "date", "produccion_kg",
            "produccion_roll_mean_3", "produccion_roll_mean_7",
            "produccion_roll_std_3", "produccion_roll_std_7"
        ]
    ].head(15)
)


,cow_id,date,produccion_kg,produccion_roll_mean_3,produccion_roll_mean_7,produccion_roll_std_3,produccion_roll_std_7
0,1204,2025-01-01,16.68,NaN,NaN,NaN,NaN
1,1204,2025-01-02,17.91,NaN,NaN,NaN,NaN
2,1204,2025-01-03,25.36,NaN,NaN,NaN,NaN
3,1204,2025-01-04,16.71,19.983333,NaN,4.696768,NaN
4,1204,2025-01-05,25.04,19.993333,NaN,4.686239,NaN
5,1204,2025-01-06,10.39,22.370000,NaN,4.904314,NaN
6,1204,2025-01-07,23.96,17.380000,NaN,7.347945,NaN
7,1204,2025-01-08,24.79,19.796667,19.435714,8.164290,5.570018
8,1204,2025-01-09,32.44,19.713333,20.594286,8.084902,5.742081
9,1204,2025-01-10,28.80,27.063333,22.670000,4.674787,7.080306


### Comentarios y observaciones

Estas variables resumen el comportamiento reciente de cada vaca:

- media reciente
- estabilidad o variabilidad reciente

Suelen ser muy útiles para capturar tendencia y volatilidad.


## 12. Variables derivadas de los cuartos de la ubre

Las variables `di`, `dd`, `ti` y `td` no deben interpretarse como variables independientes de tiempo o de ordeño.

En este contexto, su interpretación más probable es:

- `di`: delantera izquierda
- `dd`: delantera derecha
- `ti`: trasera izquierda
- `td`: trasera derecha

Es decir, representan producción o medición por **cuarto de la ubre**.

### ¿Por qué importa esto?

Porque no conviene mezclarlas como si fueran conceptos distintos.  
Más bien, tiene sentido construir variables agregadas y razones fisiológicamente interpretables, por ejemplo:

- producción total de los cuatro cuartos
- producción de cuartos delanteros
- producción de cuartos traseros
- balance izquierda / derecha
- balance traseros / delanteros


In [12]:
quarter_cols = ["di", "dd", "ti", "td"]

if set(quarter_cols).issubset(df.columns):
    df["produccion_total_cuartos"] = df["di"] + df["dd"] + df["ti"] + df["td"]

    df["produccion_delanteros"] = df["di"] + df["dd"]
    df["produccion_traseros"] = df["ti"] + df["td"]

    df["produccion_izquierda"] = df["di"] + df["ti"]
    df["produccion_derecha"] = df["dd"] + df["td"]

    df["ratio_traseros_delanteros"] = (
        df["produccion_traseros"] /
        df["produccion_delanteros"].replace(0, np.nan)
    )

    df["ratio_izquierda_derecha"] = (
        df["produccion_izquierda"] /
        df["produccion_derecha"].replace(0, np.nan)
    )

    df["diff_cuartos_vs_total"] = (
        df["produccion_total_cuartos"] - df["produccion_kg"]
    )

    display(
        df[
            [
                "cow_id", "date", "produccion_kg",
                "di", "dd", "ti", "td",
                "produccion_total_cuartos",
                "produccion_delanteros",
                "produccion_traseros",
                "produccion_izquierda",
                "produccion_derecha",
                "ratio_traseros_delanteros",
                "ratio_izquierda_derecha",
                "diff_cuartos_vs_total"
            ]
        ].head(10)
    )

    print("Resumen de diferencia entre suma de cuartos y produccion_kg:")
    display(df["diff_cuartos_vs_total"].describe())


,cow_id,date,produccion_kg,di,dd,ti,td,produccion_total_cuartos,produccion_delanteros,produccion_traseros,produccion_izquierda,produccion_derecha,ratio_traseros_delanteros,ratio_izquierda_derecha,diff_cuartos_vs_total
0,1204,2025-01-01,16.68,5.16,4.64,0.00,6.88,16.68,9.80,6.88,5.16,11.52,0.702041,0.447917,0.000000e+00
1,1204,2025-01-02,17.91,2.65,4.44,5.75,5.07,17.91,7.09,10.82,8.40,9.51,1.526093,0.883281,0.000000e+00
2,1204,2025-01-03,25.36,6.26,6.52,2.77,9.81,25.36,12.78,12.58,9.03,16.33,0.984351,0.552970,0.000000e+00
3,1204,2025-01-04,16.71,4.13,3.89,2.96,5.73,16.71,8.02,8.69,7.09,9.62,1.083541,0.737006,0.000000e+00
4,1204,2025-01-05,25.04,5.85,5.94,4.85,8.40,25.04,11.79,13.25,10.70,14.34,1.123834,0.746165,0.000000e+00
5,1204,2025-01-06,10.39,1.80,3.24,1.35,4.00,10.39,5.04,5.35,3.15,7.24,1.061508,0.435083,0.000000e+00
6,1204,2025-01-07,23.96,4.74,6.27,4.31,8.64,23.96,11.01,12.95,9.05,14.91,1.176203,0.606975,0.000000e+00
7,1204,2025-01-08,24.79,5.09,6.32,4.74,8.64,24.79,11.41,13.38,9.83,14.96,1.172656,0.657086,0.000000e+00
8,1204,2025-01-09,32.44,6.97,8.87,4.74,11.86,32.44,15.84,16.60,11.71,20.73,1.047980,0.564882,0.000000e+00
9,1204,2025-01-10,28.80,6.21,7.71,4.56,10.32,28.80,13.92,14.88,10.77,18.03,1.068966,0.597338,7.105427e-15


Resumen de diferencia entre suma de cuartos y produccion_kg:


count    9.814000e+03
mean     8.324842e-04
std      4.903709e-02
min     -1.421085e-14
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      3.450000e+00
Name: diff_cuartos_vs_total, dtype: float64

### Comentarios y observaciones

Estas variables derivadas pueden ser útiles porque capturan posibles desbalances entre zonas de la ubre.

Por ejemplo:

- una diferencia anormal entre izquierda y derecha puede sugerir un problema localizado
- una relación fuera de lo normal entre cuartos delanteros y traseros puede indicar una asimetría fisiológica o un posible problema de medición

Además, estas variables son más interpretables que usar `di`, `dd`, `ti`, `td` de forma aislada dentro del modelo.


## 13. Índice de estrés térmico (THI) a partir del clima

En lugar de usar temperatura y humedad por separado, en esta versión se construye el índice **THI** (*Temperature Humidity Index*), que resume mejor el efecto combinado del ambiente sobre el estrés térmico.

### ¿Por qué usar THI?

Porque en producción lechera el efecto del calor no depende solo de la temperatura, sino también de la humedad relativa.

Esto permite:

- resumir dos variables en una sola señal más interpretable
- reducir colinealidad entre temperatura y humedad
- capturar mejor el estrés térmico real


In [13]:
if {"temperature_c", "humidity_pct"}.issubset(df.columns):
    df["THI"] = (
        (1.8 * df["temperature_c"] + 32)
        - (
            (0.55 - 0.0055 * df["humidity_pct"])
            * ((1.8 * df["temperature_c"] + 32) - 58)
        )
    )

    df["THI_lag_1"] = df["THI"].shift(1)
    df["THI_roll_mean_3"] = df["THI"].shift(1).rolling(window=3).mean()
    df["THI_roll_mean_7"] = df["THI"].shift(1).rolling(window=7).mean()

    def clasificar_thi(thi):
        if pd.isna(thi):
            return np.nan
        if thi < 68:
            return 0
        elif thi < 72:
            return 1
        elif thi < 80:
            return 2
        else:
            return 3

    df["nivel_estres_thi"] = df["THI"].apply(clasificar_thi)

    display(df[["temperature_c", "humidity_pct", "THI", "THI_lag_1", "THI_roll_mean_3", "THI_roll_mean_7", "nivel_estres_thi"]].head(10))


,temperature_c,humidity_pct,THI,THI_lag_1,THI_roll_mean_3,THI_roll_mean_7,nivel_estres_thi
0,14.266667,43.291667,57.779807,NaN,NaN,NaN,0
1,13.404167,64.916667,56.488814,57.779807,NaN,NaN,0
2,12.791667,73.000000,55.466788,56.488814,NaN,NaN,0
3,14.333333,70.750000,57.832175,55.466788,56.578470,NaN,0
4,14.333333,58.458333,57.845696,57.832175,56.595926,NaN,0
5,12.433333,67.750000,55.022097,57.845696,57.048219,NaN,0
6,13.237500,70.666667,56.177997,55.022097,56.899989,NaN,0
7,12.529167,79.750000,54.936465,56.177997,56.348597,56.659053,0
8,13.800000,70.000000,57.031400,54.936465,55.378853,56.252862,0
9,14.604167,67.166667,58.235582,57.031400,56.048621,56.330374,0


### Comentarios y observaciones

El THI suele ser más informativo que temperatura y humedad por separado.

Además, sus rezagos y promedios móviles permiten capturar:

- estrés térmico del día anterior
- acumulación reciente de calor
- posibles efectos retardados sobre la producción


## 14. Eliminación de temperatura y humedad después de derivar THI

Una vez construido el THI, se eliminan `temperature_c` y `humidity_pct` para dejar una representación climática más compacta y menos redundante.


In [14]:
drop_weather_cols = [
    "temperature_c",
    "humidity_pct"
]

drop_weather_cols = [c for c in drop_weather_cols if c in df.columns]

df = df.drop(columns=drop_weather_cols)

print("Columnas climáticas eliminadas:", drop_weather_cols)
print("Shape actual:", df.shape)


Columnas climáticas eliminadas: ['temperature_c', 'humidity_pct']
Shape actual: (9814, 60)


### Comentarios y observaciones

Esta decisión ayuda a:

- reducir redundancia
- disminuir colinealidad
- simplificar el conjunto de variables del modelo base

Por ahora se conservan otras variables climáticas como `pressure_msl`, `rain_mm` y `wind_speed`, ya que pueden aportar señal complementaria.


## 15. Variables relativas por vaca

Como cada vaca tiene un nivel base distinto de producción, puede ser útil construir variables centradas respecto a su promedio histórico global.


In [15]:
cow_mean_prod = df.groupby("cow_id")["produccion_kg"].transform("mean")
cow_std_prod = df.groupby("cow_id")["produccion_kg"].transform("std")

df["produccion_relativa_vaca"] = df["produccion_kg"] - cow_mean_prod
df["produccion_zscore_vaca"] = (df["produccion_kg"] - cow_mean_prod) / cow_std_prod.replace(0, np.nan)

display(
    df[
        ["cow_id", "date", "produccion_kg", "produccion_relativa_vaca", "produccion_zscore_vaca"]
    ].head(15)
)


,cow_id,date,produccion_kg,produccion_relativa_vaca,produccion_zscore_vaca
0,1204,2025-01-01,16.68,-10.104345,-1.283285
1,1204,2025-01-02,17.91,-8.874345,-1.127071
2,1204,2025-01-03,25.36,-1.424345,-0.180897
3,1204,2025-01-04,16.71,-10.074345,-1.279475
4,1204,2025-01-05,25.04,-1.744345,-0.221538
5,1204,2025-01-06,10.39,-16.394345,-2.082136
6,1204,2025-01-07,23.96,-2.824345,-0.358701
7,1204,2025-01-08,24.79,-1.994345,-0.253288
8,1204,2025-01-09,32.44,5.655655,0.718287
9,1204,2025-01-10,28.80,2.015655,0.255995


### Comentarios y observaciones

Estas variables ayudan a separar:

- nivel base de cada vaca
- desviaciones respecto a su comportamiento típico

Esto puede mejorar bastante el desempeño de algunos modelos.


## 16. Revisión rápida del dataset enriquecido

Se inspeccionan dimensiones, columnas nuevas y proporción de faltantes después de la ingeniería de variables.


In [16]:
print("Shape final:", df.shape)

base_cols = [
    "cow_id", "date", "produccion_kg", "di", "dd", "td", "ti",
    "ordenos_dia", "duracion_total_min", "pressure_msl", "rain_mm", "wind_speed",
    "month", "year", "day",
    "weekday", "weekday_date", "eventos_pdf_count", "eventos_pdf_text",
    "rumia_min", "days_in_milk", "lactation_age", "group_id"
]

new_cols = [col for col in df.columns if col not in base_cols]

print("Número de columnas nuevas aproximado:", len(new_cols))
display(new_cols[:50])

missing_final = df.isna().mean().sort_values(ascending=False)
display(missing_final.head(25))


Shape final: (9814, 62)
Número de columnas nuevas aproximado: 39


['ubre',
 'destino_leche',
 'ms',
 'source_file',
 'anio',
 'mes',
 'dia_mes',
 'dia_semana',
 'es_fin_de_semana',
 'dia_del_anio',
 'mes_sin',
 'mes_cos',
 'dia_semana_sin',
 'dia_semana_cos',
 'produccion_lag_1',
 'produccion_lag_2',
 'produccion_lag_3',
 'produccion_lag_7',
 'delta_produccion_1d',
 'delta_produccion_3d',
 'produccion_roll_mean_3',
 'produccion_roll_mean_7',
 'produccion_roll_std_3',
 'produccion_roll_std_7',
 'produccion_total_cuartos',
 'produccion_delanteros',
 'produccion_traseros',
 'produccion_izquierda',
 'produccion_derecha',
 'ratio_traseros_delanteros',
 'ratio_izquierda_derecha',
 'diff_cuartos_vs_total',
 'THI',
 'THI_lag_1',
 'THI_roll_mean_3',
 'THI_roll_mean_7',
 'nivel_estres_thi',
 'produccion_relativa_vaca',
 'produccion_zscore_vaca']

rumia_min                    0.997249
days_in_milk                 0.997147
group_id                     0.997147
weekday                      0.997147
lactation_age                0.997147
eventos_pdf_text             0.984920
eventos_pdf_count            0.984920
produccion_roll_std_7        0.046362
produccion_lag_7             0.046362
produccion_roll_mean_7       0.046362
produccion_lag_3             0.019870
delta_produccion_3d          0.019870
produccion_roll_mean_3       0.019870
produccion_roll_std_3        0.019870
produccion_lag_2             0.013246
delta_produccion_1d          0.006623
produccion_lag_1             0.006623
THI_roll_mean_7              0.000713
THI_roll_mean_3              0.000306
ratio_izquierda_derecha      0.000306
ratio_traseros_delanteros    0.000204
THI_lag_1                    0.000102
source_file                  0.000000
td                           0.000000
ti                           0.000000
dtype: float64

### Comentarios y observaciones

Es normal que las variables con rezago y rolling tengan faltantes al inicio de cada serie por vaca.

Estos faltantes no necesariamente son un problema; más adelante se pueden:

- eliminar filas iniciales
- imputar
- o dejar que el pipeline del modelo los trate explícitamente


## 17. Selección preliminar de variables para un modelo base

Aquí se define una lista preliminar de variables candidatas para un primer modelo.

Esta lista se puede ajustar más adelante según:

- cobertura
- importancia de variables
- colinealidad
- resultados de validación


In [17]:
candidate_features = [
    "di", "dd", "td", "ti", "ordenos_dia", "duracion_total_min",
    "pressure_msl", "rain_mm", "wind_speed",
    "anio", "mes", "dia_mes", "dia_semana", "es_fin_de_semana", "dia_del_anio",
    "mes_sin", "mes_cos", "dia_semana_sin", "dia_semana_cos",
    "produccion_lag_1", "produccion_lag_2", "produccion_lag_3", "produccion_lag_7",
    "delta_produccion_1d", "delta_produccion_3d",
    "produccion_roll_mean_3", "produccion_roll_mean_7",
    "produccion_roll_std_3", "produccion_roll_std_7",
    "produccion_total_cuartos", "produccion_delanteros", "produccion_traseros",
    "produccion_izquierda", "produccion_derecha",
    "ratio_traseros_delanteros", "ratio_izquierda_derecha",
    "THI", "THI_lag_1", "THI_roll_mean_3", "THI_roll_mean_7", "nivel_estres_thi",
    "produccion_relativa_vaca", "produccion_zscore_vaca"
]

candidate_features = [col for col in candidate_features if col in df.columns]

print("Número de variables candidatas:", len(candidate_features))
display(candidate_features)


Número de variables candidatas: 43


['di',
 'dd',
 'td',
 'ti',
 'ordenos_dia',
 'duracion_total_min',
 'pressure_msl',
 'rain_mm',
 'wind_speed',
 'anio',
 'mes',
 'dia_mes',
 'dia_semana',
 'es_fin_de_semana',
 'dia_del_anio',
 'mes_sin',
 'mes_cos',
 'dia_semana_sin',
 'dia_semana_cos',
 'produccion_lag_1',
 'produccion_lag_2',
 'produccion_lag_3',
 'produccion_lag_7',
 'delta_produccion_1d',
 'delta_produccion_3d',
 'produccion_roll_mean_3',
 'produccion_roll_mean_7',
 'produccion_roll_std_3',
 'produccion_roll_std_7',
 'produccion_total_cuartos',
 'produccion_delanteros',
 'produccion_traseros',
 'produccion_izquierda',
 'produccion_derecha',
 'ratio_traseros_delanteros',
 'ratio_izquierda_derecha',
 'THI',
 'THI_lag_1',
 'THI_roll_mean_3',
 'THI_roll_mean_7',
 'nivel_estres_thi',
 'produccion_relativa_vaca',
 'produccion_zscore_vaca']

### Comentarios y observaciones

Esta selección no es definitiva.  
Es solo una base razonable para comenzar a experimentar con modelos.


## 18. Guardado del dataset enriquecido

Finalmente se guarda el dataset con ingeniería de variables para usarlo en la siguiente etapa del pipeline.


In [ ]:
OUTPUT_PATH = "../data/processed/training_dataset_features.parquet"

df.to_parquet(OUTPUT_PATH, index=False)
print("Archivo guardado en:")
print(OUTPUT_PATH)

# también exportar a csv
CSV_PATH = OUTPUT_PATH.replace(".parquet", ".csv")
df.to_csv(CSV_PATH, index=False)
print("Archivo CSV guardado en:")
print(CSV_PATH)


Archivo guardado en:
Archivo guardado en:
../data/processed/training_dataset_features.parquet
Archivo CSV guardado en:
../data/processed/training_dataset_features.csv


## 19. Conclusiones de la ingeniería de variables

En esta etapa se logró:

- ordenar y normalizar el dataset base
- construir variables de calendario
- crear variables cíclicas
- generar rezagos de producción
- generar promedios móviles
- construir el índice THI a partir del clima
- reemplazar temperatura y humedad por una representación más útil del estrés térmico
- derivar variables más interpretables a partir de los cuartos de la ubre
- obtener una primera lista de variables candidatas para modelado

### Conclusión general

El dataset ahora tiene una estructura más útil para entrenamiento de modelos que la versión original, especialmente porque ya incorpora:

- variables temporales por vaca
- una medida climática más adecuada para producción lechera
- variables agregadas con interpretación fisiológica de la ubre

Los siguientes pasos naturales son:

1. definir conjunto final de variables
2. separar train / validation / test respetando el tiempo
3. entrenar modelos base
4. comparar desempeño con y sin variables temporales
5. evaluar el impacto del THI y de las variables derivadas de los cuartos de la ubre
